# Handling missing values with SKLearn

In [5]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

In [6]:
car_sales_missing_df = pd.read_csv("./data/car-sales-extended-missing-data.csv")
car_sales_missing_df.head()

,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431.0,4.0,15323.0
1,BMW,Blue,192714.0,5.0,19943.0
2,Honda,White,84714.0,4.0,28343.0
3,Toyota,White,154365.0,4.0,13434.0
4,Nissan,Blue,181577.0,3.0,14043.0


In [7]:
car_sales_missing_df.isna().sum()

Make             49
Colour           50
Odometer (KM)    50
Doors            50
Price            50
dtype: int64

In [8]:
car_sales_missing_df.dropna(subset = ["Price"], inplace=True)
car_sales_missing_df.isna().sum()

Make             47
Colour           46
Odometer (KM)    48
Doors            47
Price             0
dtype: int64

In [9]:
X = car_sales_missing_df.drop("Price", axis = 1)
y = car_sales_missing_df["Price"]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### Handling missing values

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour"]
doors_feature = ["Doors"]
numerical_features = ["Odometer (KM)"]

categorical_imputer = SimpleImputer(strategy = "constant", fill_value = "Missing")
doors_imputer = SimpleImputer(strategy = "constant", fill_value = 4)
numerical_imputer = SimpleImputer(strategy = "mean")

preprecessor = ColumnTransformer(transformers = [
    ("cat", categorical_imputer, categorical_features),
    ("doors", doors_imputer, doors_feature),
    ("num", numerical_imputer, numerical_features)
], remainder="passthrough")

imputed_X_train = preprecessor.fit_transform(X_train)
print(type(imputed_X_train))
imputed_X_test = preprecessor.transform(X_test)
print(type(imputed_X_test))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [11]:
imputed_X_train_df = pd.DataFrame(imputed_X_train, columns = X_train.columns)
imputed_X_train_df.head()

,Make,Colour,Odometer (KM),Doors
0,Toyota,White,4.0,73869.0
1,Missing,Black,4.0,224479.0
2,Nissan,Blue,4.0,104355.0
3,Honda,Blue,4.0,71949.0
4,Toyota,Blue,4.0,244204.0


In [12]:
imputed_X_test_df = pd.DataFrame(imputed_X_test, columns = X_test.columns)
imputed_X_test_df.head()

,Make,Colour,Odometer (KM),Doors
0,Nissan,White,4.0,49310.0
1,Toyota,White,4.0,146824.0
2,BMW,White,5.0,178796.0
3,Honda,Green,4.0,165101.0
4,Honda,Red,4.0,86427.0


In [13]:
imputed_X_train_df.isna().sum()

Make             0
Colour           0
Odometer (KM)    0
Doors            0
dtype: int64

In [14]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
encoder = OneHotEncoder(handle_unknown = "ignore")
scaler = StandardScaler()

preprocessor = ColumnTransformer([
    ("cat", encoder, [*categorical_features, *doors_feature]),
    ("num", scaler, numerical_features)
], remainder = "passthrough")

transformed_X_train = preprocessor.fit_transform(imputed_X_train_df)
transformed_X_test = preprocessor.transform(imputed_X_test_df)
transformed_X_train.toarray()

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.01737463],
       [ 0.        ,  0.        ,  1.        , ...,  0.        ,
         0.        , -0.01737463],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.01737463],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.01737463],
       [ 0.        ,  1.        ,  0.        , ...,  0.        ,
         0.        , -0.01737463],
       [ 0.        ,  1.        ,  0.        , ...,  0.        ,
         0.        , -0.01737463]])

In [15]:
from sklearn.ensemble import RandomForestRegressor
rg = RandomForestRegressor(random_state=42)
rg.fit(transformed_X_train, y_train)

RandomForestRegressor(random_state=42)

In [16]:
rg.score(transformed_X_test, y_test)

-0.007553928073975369

### Choose the right estimator / model for our problems

![](./images/estimators.png)

In [17]:
# Picking a machine learning model for a regression problem ----- California Housing dataset 

In [ ]:
from sklearn.datasets import fetch_california_housing
california_housing_dataset = fetch_california_housing()

In [24]:
house_data = pd.DataFrame(california_housing_dataset.data, columns=california_housing_dataset.feature_names)
house_data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [26]:
california_housing_dataset.target_names

['MedHouseVal']

In [30]:
house_data["Target"] = california_housing_dataset.target

In [31]:
house_data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,Target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,3.422


In [ ]:
house_data = house_data.drop(["MedHouseVal"], axis=1)

In [39]:
house_data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [47]:
# Import estimator
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
model = Ridge(random_state=0)

# Create the data
X = house_data.drop("Target", axis = 1)
y = house_data["Target"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Fit the model
model.fit(X_train, y_train)

# Evaluate a model
model.score(X_test, y_test)

0.6070423850012195